In [1]:
!pip install faiss-cpu
!pip install numpy==1.26.4


In [2]:
import torch
import numpy as np
import os
import faiss # Import faiss
from datasets import Dataset, load_from_disk
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration, DPRContextEncoder

In [23]:
# Step 1: Load scraped documents
corpus = []
folder_path = "/content/Research-Chatbot"

for file_name in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file_name)
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            corpus.extend([para.strip() for para in content.split('\n') if len(para.strip()) > 10])
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print(f"Loaded {len(corpus)} paragraphs.")

Error reading /content/Research-Chatbot/.ipynb_checkpoints: [Errno 21] Is a directory: '/content/Research-Chatbot/.ipynb_checkpoints'
Loaded 67 paragraphs.


In [28]:
print(corpus)


['"Postgraduate Programmes","Dublin Business School (DBS) specialises in the provision of career-focused business and law education as well as the delivery of contemporary programmes in the areas of creative media, marketing, counselling & psychotherapy, humanities and psychology.\\n\\nVisit our website: www.dbs.ie",', '"Why Study for a Postgraduate Qualification?","To compliment your primary degree\\n\\nTo fulfil personal interests in a particular subject area\\n\\nPromotion opportunities in current role/career enhancement\\n\\nEnhanced job security\\n\\nDevelopment of transferable skills\\n\\nWorthwhile investment to enhance lifetime career prospects\\n\\nAcademic and personal development at a professional level\\n\\nFurther development of your skills and knowledge in a specialist field\\n\\nChange of direction in career\\n\\nImproved career and networking prospects\\n\\nOpportunity to develop strong analytical, problem-solving, presentation, time management and communication skills"

67

In [13]:
# Step 2: Create Hugging Face Dataset
data_dict = {"text": corpus, "title": [""] * len(corpus)}  # Dummy title
dbs_dataset = Dataset.from_dict(data_dict)

In [27]:
dbs_dataset.data

InMemoryTable
text: string
title: string
----
text: [[""Postgraduate Programmes","Dublin Business School (DBS) specialises in the provision of career-focused business and law education as well as the delivery of contemporary programmes in the areas of creative media, marketing, counselling & psychotherapy, humanities and psychology.\n\nVisit our website: www.dbs.ie",",""Why Study for a Postgraduate Qualification?","To compliment your primary degree\n\nTo fulfil personal interests in a particular subject area\n\nPromotion opportunities in current role/career enhancement\n\nEnhanced job security\n\nDevelopment of transferable skills\n\nWorthwhile investment to enhance lifetime career prospects\n\nAcademic and personal development at a professional level\n\nFurther development of your skills and knowledge in a specialist field\n\nChange of direction in career\n\nImproved career and networking prospects\n\nOpportunity to develop strong analytical, problem-solving, presentation, time manage

In [7]:
# Step 3: Load DPR Context Encoder & Tokenizer
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq")

# Step 4: Generate embeddings
def get_embeddings(example):
    inputs = tokenizer(example["text"], return_tensors="pt", truncation=True, padding="max_length", max_length=512)
    with torch.no_grad():
        outputs = context_encoder(**inputs)
        # Return NumPy array directly
        return {"embeddings": outputs.pooler_output[0].detach().cpu().numpy()}


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-ctx_encoder-single-nq-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizerFast'.


In [8]:
# Step 5: Format column for FAISS indexing

dbs_dataset = dbs_dataset.map(get_embeddings)
print("Columns in dataset:", dbs_dataset.column_names)

dbs_dataset.set_format(type="numpy", columns=["embeddings"])
embedding = np.asarray(dbs_dataset[0]["embeddings"])
print(embedding.shape)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Map:   0%|          | 0/44986 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
dbs_dataset

In [ ]:
# Step 6: Add FAISS index
if "embeddings" in dbs_dataset.list_indexes():
    dbs_dataset.drop_index("embeddings")

embedding_matrix = np.stack([np.asarray(d["embeddings"]) for d in dbs_dataset], axis=0).astype("float32")
dimension = embedding_matrix.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)  # L2 = Euclidean distance
faiss_index.add(embedding_matrix)
dbs_dataset.add_faiss_index(column="embeddings")

In [ ]:
# Step 7: Save the dataset to disk
index_path = "/content/faiss_index.index"
dataset_path = "/content/dbs_dataset"

#dbs_dataset.add_faiss_index(column="embeddings")
dbs_dataset.save_faiss_index("embeddings", index_path)
dbs_dataset.drop_index("embeddings")
dbs_dataset.save_to_disk(dataset_path)

In [ ]:
# Step 8: Load for Retrieval
# Load dataset
dbs_dataset = load_from_disk(dataset_path)
# Reload FAISS index
dbs_dataset.load_faiss_index("embeddings", index_path)

In [ ]:

# Step 9: Load RAG Retriever
retriever = RagRetriever.from_pretrained(
    "facebook/rag-token-nq",
    index_name="custom",
    passages_path=dataset_path,
    index_path=index_path,
    indexed_dataset=dbs_dataset
)

# Step 10: Load RAG Model
model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-nq", retriever=retriever)

# Step 11: Define chatbot function
def rag_chatbot(query):
    print(query)
    inputs = tokenizer(query, return_tensors="pt")
    generated = model.generate(
        input_ids=inputs["input_ids"],
        num_return_sequences=1,
        num_beams=1,
        max_length=50
    )
    return tokenizer.decode(generated[0], skip_special_tokens=True)



In [ ]:
# Example usage
response = rag_chatbot("DBS ratings did DBS earned?")
print("Bot:", response)

In [ ]:
input_list = ["How many books DBS Library has?",
              "Where can I access DBS WiFi?",
              "What are library opening hours?",
              "What ratings did DBS earned?",
              "how to view Library account?",
              "Are the Guides to Library resources for students with disabilities are also available in the Library?",
              "How many university partnerships does DBS has developed?"
             ]

# create a function which takes a list of queries and output an object of list with quer and response

def provide_set_of_responses(input_list):
    output_table = []
    for query in input_list:
        response = rag_chatbot(query)
        output_table.append({"query": query , "expected_answer": response})
    return output_table


out = provide_set_of_responses(input_list)
print(out)


In [ ]:
print("Corpus size:", len(corpus))
print("Dataset size:", len(dbs_dataset))


Evaluation Framework

In [ ]:
!pip install nltk

In [ ]:
!pip install bert-score

In [ ]:
import nltk
nltk.download('punkt')


In [ ]:
test_set = [
    {"query": "How many books DBS Library has?", "expected_answer": "over 43,000"},
    {"query": "Where can I access DBS WiFi?", "expected_answer": "DBS WiFi is available throughout all campuses."},
    {"query": "What are library opening hours?", "expected_answer": "The library is open Monday to Friday from 9am to 8pm."},
    {"query": "What ratings did DBS earned?", "expected_answer": "DBS earned an overall rating of 4 Stars"}

]


test_set = [
    {"query": "How many books DBS Library has?", "expected_answer": " over 43,000"},
     {"query": "Where can I access DBS WiFi?", "expected_answer": ""},
      {"query": "What are library opening hours?", "expected_answer": " 24 hours a day"},
       {"query": "What ratings did DBS earned?", "expected_answer": " 4 stars"},
        {"query": "how to view Library account?", "expected_answer": ""},
         {"query": "Are the Guides to Library resources for students with disabilities are also available in the Library?", "expected_answer": " on the library website"},
          {"query": "How many university partnerships does DBS has developed?", "expected_answer": " over 75"}
  ]


In [ ]:
import time
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score

smoothie = SmoothingFunction().method4

def evaluate_rag_model(test_set):
    results = []
    total_time = 0
    all_generated = []
    all_expected = []

    for i, item in enumerate(test_set):
        query = item["query"]
        expected = item["expected_answer"]

        start_time = time.time()
        generated = rag_chatbot(query)
        end_time = time.time()

        response_time = end_time - start_time
        total_time += response_time

        # Save for BERTScore later
        all_generated.append(generated)
        all_expected.append(expected)

        # Exact match
        exact_match = int(expected.lower() in generated.lower())

        # BLEU Score
        reference = [expected.split()]
        candidate = generated.split()
        bleu = sentence_bleu(reference, candidate, smoothing_function=smoothie)

        results.append({
            "Query": query,
            "Generated": generated,
            "Expected": expected,
            "ExactMatch": exact_match,
            "BLEU": bleu,
            "TimeTaken": response_time
        })

    # BERTScore
    P, R, F1 = bert_score(all_generated, all_expected, lang="en", verbose=True)
    avg_bertscore_f1 = F1.mean().item()

    # Summary metrics
    accuracy = sum(r["ExactMatch"] for r in results) / len(results)
    avg_bleu = sum(r["BLEU"] for r in results) / len(results)
    avg_time = total_time / len(results)

    print(f"\n--- Evaluation Summary ---")
    print(f"Accuracy (Exact Match): {accuracy:.2f}")
    print(f"Average BLEU Score: {avg_bleu:.2f}")
    print(f"Average BERTScore F1: {avg_bertscore_f1:.2f}")
    print(f"Average Inference Time: {avg_time:.2f} seconds\n")

    return results


In [ ]:
results = evaluate_rag_model(test_set)

In [ ]:
results